In [ ]:
# External test on IDRiD (Indian Diabetic Retinopathy Image Dataset, ISBI 2018; CC BY-NC 4.0 copy on Kaggle). v2:
# the training and testing folders reuse file names (IDRiD_001...), so each folder is prepared separately.
import glob, os, shutil
import pandas as pd
src = os.path.dirname(os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/runs/b4_512/best.pt", recursive=True)[0])))
shutil.copytree(src, "/kaggle/working/dr-baseline", dirs_exist_ok=True)
os.chdir("/kaggle/working/dr-baseline")
GT_TRAIN = glob.glob("/kaggle/input/**/2. Groundtruths/a.*.csv", recursive=True)[0]
GT_TEST = glob.glob("/kaggle/input/**/2. Groundtruths/b.*.csv", recursive=True)[0]
IMG_TRAIN = glob.glob("/kaggle/input/**/1. Original Images/a. Training Set", recursive=True)[0]
IMG_TEST = glob.glob("/kaggle/input/**/1. Original Images/b. Testing Set", recursive=True)[0]
for part, gt in (("train", GT_TRAIN), ("test", GT_TEST)):
    lab = pd.read_csv(gt).iloc[:, :3]
    lab.columns = ["Image name", "Retinopathy grade", "Risk of macular edema"]
    lab = lab.dropna()
    lab.to_csv(f"/kaggle/working/idrid_{part}.csv", index=False)
    print(part, len(lab), "photos, grades:", lab["Retinopathy grade"].value_counts().sort_index().to_dict())

In [ ]:
!python prepare_data.py external --name idrid_train --csv /kaggle/working/idrid_train.csv --image-dir "{IMG_TRAIN}" --image-col "Image name" --label-col "Retinopathy grade" --dme-col "Risk of macular edema" --out-dir data/processed
!python prepare_data.py external --name idrid_test --csv /kaggle/working/idrid_test.csv --image-dir "{IMG_TEST}" --image-col "Image name" --label-col "Retinopathy grade" --dme-col "Risk of macular edema" --out-dir data/processed

In [ ]:
# All 516 photos, with the threshold locked on APTOS validation (no IDRiD tuning)
pd.concat([pd.read_csv("data/processed/external_idrid_train.csv"), pd.read_csv("data/processed/external_idrid_test.csv")]).to_csv("data/processed/external_idrid_all.csv", index=False)
!python evaluate.py --run-dir runs/b4_512 --csv data/processed/external_idrid_all.csv --n-boot 1000

In [ ]:
# Site re-tuning, done fairly: choose the threshold on IDRiD's training folder, measure it on IDRiD's testing folder
!python evaluate.py --run-dir runs/b4_512 --csv data/processed/external_idrid_train.csv --n-boot 0
!python evaluate.py --run-dir runs/b4_512 --csv data/processed/external_idrid_test.csv --n-boot 1000
from dr.metrics import threshold_for_sensitivity, binary_metrics, roc_auc, group_bootstrap
tr = pd.read_csv("runs/b4_512/eval_external_idrid_train/predictions.csv")
te = pd.read_csv("runs/b4_512/eval_external_idrid_test/predictions.csv")
site_t = threshold_for_sensitivity(tr["referable"], tr["prob_referable"], 0.90)
y, p = te["referable"].to_numpy(), te["prob_referable"].to_numpy()
ci = group_bootstrap(lambda i: binary_metrics(y[i], p[i], site_t), te["image_id"].astype(str), n_boot=1000)
m = binary_metrics(y, p, site_t)
print(f"SITE-TUNED threshold {site_t:.3f} (APTOS threshold 0.555). IDRiD testing folder, {len(te)} photos:")
print(f"  sensitivity {m['sensitivity']:.3f} (95% CI {ci['sensitivity'][0]:.3f}-{ci['sensitivity'][1]:.3f}) | specificity {m['specificity']:.3f} (95% CI {ci['specificity'][0]:.3f}-{ci['specificity'][1]:.3f})")

In [ ]:
!python rim_test.py --run-dir runs/b4_512 --csv data/processed/external_idrid_all.csv --split external --out runs/b4_512/rim_test_idrid

In [ ]:
!python explain_examples.py --run-dir runs/b4_512 --csv data/processed/external_idrid_all.csv --predictions runs/b4_512/eval_external_idrid_all/predictions.csv --out runs/b4_512/gradcam_idrid.jpg

In [ ]:
!rm -rf data/processed/images; du -sh /kaggle/working; find /kaggle/working -type f | wc -l